In [1]:
import os
os.environ["NPU_VISIBLE_DEVICES"]="5"
os.environ["ASCEND_RT_VISIBLE_DEVICES"]="5"
import json
from typing import Dict, List, Any
from tqdm import tqdm
from functools import partial

import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch_npu
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC2/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


In [2]:
base_model = "/data/pretrained-models/meta/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [3]:
base_dataset = "/data/datasets/Llama-3.2-3B-Instruct-evals"
general_datasets = [
    "Llama-3.2-3B-Instruct-evals__mmlu__details",
    "/data/datasets/ultrachat_200k",
]
reason_datasets = [
    "Llama-3.2-3B-Instruct-evals__gpqa__details",
    "Llama-3.2-3B-Instruct-evals__arc_challenge__details"
]
math_datasets = [
    # "Llama-3.2-3B-Instruct-evals__gsm8k__details",
    "gsm8k",
    "/data/datasets/MathInstruct",
    "Llama-3.2-3B-Instruct-evals__math__details"
]
humaneval_datasets = [
    "evalplus/humanevalplus",
]
magicoder_datasets = [
    "/data/datasets/Magicoder-Evol-Instruct-110K",
]
mbpp_datasets = [
    "evalplus/mbppplus"
]

In [4]:
def preprocess_gsm8k(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str, Any]:
    prefix = "Given the following problem, reason and give a final answer to the problem.\nProblem: {{question}}\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.\n"
    icl = [
        {
            "role" : "user",
            "content" : "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?"
        },
        {
            "role" : "assistant",
            "content" : "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The final answer is 6"
        },
        {
            "role": "user",
            "content": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?"
        },
        {
            "role": "assistant",
            "content" : "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The final answer is 5"
        },
        {
            "role": "user",
            "content" : "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
        },
        {
            "role" : "assistant",
            "content" : "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The final answer is 39"
        },
        {
            "role" : "user",
            "content" : "Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?"
        },
        {
            "role" : "assistant",
            "content" : "Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The final answer is 8"
        },
        {
            "role" : "user",
            "content" : "Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?"
        },
        {
            "role" : "assistant",
            "content" : "Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The final answer is 9"
        },
        {
            "role" : "user",
            "content" : "There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?"
        },
        {
            "role" : "assistant",
            "content" : "There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The final answer is 29"
        },
        {
            "role" : "user",
            "content" : "Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?"
        },
        {
            "role" : "assistant",
            "content" : "Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The final answer is 33"
        },
        {
            "role" : "user",
            "content" : "Olivia has $23. She bought five bagels for $3 each. How much money does she have left?"
        },
        {
            "role" : "assistant",
            "content" : "Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The final answer is 8"
        }
    ]
    for i in range(len(icl)):
        if icl[i]['role'] == "user":
            icl[i]['content'] = prefix.replace("{{question}}", icl[i]['content'])
    icl.append({
        "role": "user",
        "content": prefix.replace("{{question}}", examples["question"].strip())
    })
    messages = tokenizer.apply_chat_template(icl, tokenize=False, add_generation_prompt=True)
    return { "inst": messages }

def preprocess_ultrachat(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str,Any]:
    messages = examples['messages']
    # messages = tokenizer.apply_chat_template(messages, tokenize=False)
    # return { "inst":messages }
    messages = tokenizer.apply_chat_template(messages[:-1] if len(messages) > 1 else messages, tokenize=False, add_generation_prompt=True)
    return { "inst":messages }

In [5]:
# def preprocess_fn_general(example:Dict[str, Any], tokenizer:AutoTokenizer):
#     multi_turns = tokenizer.encode(example['input_final_prompts'][0])
#     return { "input_ids": [multi_turns] }

def task_preprocess(example:Dict[str, str], tokenizer:AutoTokenizer, task:str="humaneval")->Dict[str, str]:
    if task == "humaneval":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        task_prompt = f"""\
{instruction_prefix}
```
{example['prompt'].strip()}
```
"""
        response = f"""\
{response_prefix}
```python
{_MAGIC_SPLITTER_}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "mbpp":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
#         task_prompt = f"""\
# {example['prompt'].strip()}
# {example['test_list'][0].strip()}
# {instruction_prefix}
# ```python
# ......
# ```
# """
#         response = f"""\
# {response_prefix}
# ```python
# {_MAGIC_SPLITTER_}
# ```
# """
        python_prefix = 'Write a python function to '
        func_prefix = 'Write a function to '
        if python_prefix in example['prompt']:
            prefix = python_prefix
        elif func_prefix in example['prompt']:
            prefix = func_prefix
        else:
            prefix = ""
        prompt = example['prompt'].replace(prefix, '').strip().capitalize()
        task_prompt = f"""\
{instruction_prefix}
```
{example['code'].split(":")[0].strip()}:
    \"\"\"
    {prompt}
    >>> {example['test_list'][0].replace("assert", "").strip()}
    True
    \"\"\"
```
"""
        response = f"""\
{response_prefix}
```python
{_MAGIC_SPLITTER_}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "magicoder":
        return {
            "inst": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction'].strip()}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
            }
    elif task == "mathinstruct":
        return {
            "inst": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction'].strip()}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
        }

  

In [6]:
general_datasets = [
    # load_dataset(
    #     base_dataset,
    #     name=item,
    #     num_proc=8,
    # ).map(
    #     partial(preprocess_fn_general, tokenizer=tokenizer),
    #     num_proc=8,
    # )['latest'] for item  in general_datasets
    load_dataset(
        base_dataset,
        name=general_datasets[0],
        num_proc=8,
    )['latest'],
    load_dataset(
        general_datasets[1],
        num_proc=8,
    ).map(
        partial(preprocess_ultrachat, tokenizer=tokenizer),
        num_proc=8
    )['train_sft'],
]
reason_datasets = [
    load_dataset(
        base_dataset,
        name=item,
        num_proc=8,
    )['latest'] for item  in reason_datasets
]
math_datasets = [
    # load_dataset(
    #     base_dataset,
    #     name=item,
    #     num_proc=8,
    # ).map(
    #     partial(preprocess_fn_general, tokenizer=tokenizer),
    #     num_proc=8,
    # )['latest'] for item in math_datasets
    # load_dataset(
    #     base_dataset,
    #     name=math_datasets[0],
    #     num_proc=8,
    # )['latest'],
    load_dataset(
        math_datasets[0],
        "main",
        num_proc=8,
    )['train'].map(
        partial(preprocess_gsm8k, tokenizer=tokenizer),
        num_proc=8,
    ),
    load_dataset(
        math_datasets[1],
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="mathinstruct"),
        num_proc=8,
    )['train'],
    load_dataset(
        base_dataset,
        name=math_datasets[2],
        num_proc=8,
    )['latest']
]
humaneval_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="humaneval"),
        num_proc=8,
    )['test'] for item in humaneval_datasets
]
magicoder_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="magicoder"),
        num_proc=8,
        load_from_cache_file=False,
    )['train'] for item in magicoder_datasets
]
mbpp_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="mbpp"),
        num_proc=8,
        load_from_cache_file=False,
    )['test'] for item in mbpp_datasets
]

Map (num_proc=8):   0%|          | 0/111183 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/378 [00:00<?, ? examples/s]

In [7]:
# print(magicoder_datasets[0][4]['inst'])
print(mbpp_datasets[0][5]['inst'])
# print(mbpp_datasets[0][5]['code'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Please provide a self-contained Python script that solves the following problem in a markdown code block:
```
def square_nums(nums):
    """
    Find squares of individual elements in a list.
    >>> square_nums([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])==[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
    True
    """
```<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:
```python



In [8]:
print(humaneval_datasets[0][4]['inst'])
# print(humaneval_datasets[0][1]['canonical_solution'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Please provide a self-contained Python script that solves the following problem in a markdown code block:
```
from typing import List


def mean_absolute_deviation(numbers: List[float]) -> float:
    """ For a given list of input numbers, calculate Mean Absolute Deviation
    around the mean of this dataset.
    Mean Absolute Deviation is the average absolute difference between each
    element and a centerpoint (mean in this case):
    MAD = average | x - x_mean |
    >>> mean_absolute_deviation([1.0, 2.0, 3.0, 4.0])
    1.0
    """
```<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:
```python



In [9]:
# print(general_datasets[2][0]['input_final_prompts'][0])
print(general_datasets[1][200]['inst'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Write a horror story about a monster lurking in the darkness, with a focus on building suspense through vivid sensory details and psychological terror. Incorporate elements of foreshadowing and surprise twists to create a truly shocking ending. Consider playing with the reader's expectations and subverting traditional horror tropes to keep them on the edge of their seat. Additionally, utilize strong character development to create a sense of emotional investment and make the terror feel all too real.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

It was a dark and stormy night, the kind that kept even the bravest souls indoors. But Emily wasn't one to heed warnings, and she had dared to venture out into the night. Now she found herself lost, wandering through an unfamiliar forest as the rain began to pel

In [10]:
# print(math_datasets[0][0]['input_final_prompts'][0])
# print(math_datasets[0][0]['input_correct_responses'])
print(math_datasets[-1][0]['input_final_prompts'][0])

<|start_header_id|>user<|end_header_id|>

Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

Problem: Find the domain of the expression $\frac{\sqrt{x-2}}{\sqrt{5-x}}$.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

## Step 1: Consider the expression inside the first square root
The expression inside the first square root is $x-2$, and it must be non-negative. This means that $x-2 \ge 0$, which simplifies to $x \ge 2$.

## Step 2: Consider the expression insi

In [11]:
print(math_datasets[1][1]['inst'])
print(math_datasets[1][1]['output'])

<|begin_of_text|><|start_header_id|>user<|end_header_id|>

How many ways can the letters in the word COMMON be arranged?
Answer Choices: (A) 6 (B) 30 (C) 90 (D) 120 (E) 180<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Let's solve the multi-choice question step by step.
According to the above the # of permutations of 6 letters COMMON out of which 2 O's and 2 M's are identical is 6!2!∗2!=180
The answer is E.


In [12]:
# print(magicoder_datasets[0][3]['inst'])
# print(magicoder_datasets[0][3].keys())
print(magicoder_datasets[0][3]['response'])

This task requires writing of a significant volume of code, which is not fully suitable for a text-based medium. However, I will outline a general solution using Python and scikit-learn. We'll use "CountVectorizer" for bag-of-words model and "TfidVectorizer" for TF-IDF. To handle different languages, we can use 'langdetect' library.

1. Import required libraries
```python
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from langdetect import detect
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
```

2. Load sentence data and labels. For example, if data is stored in a csv format:
```python
data = pd.read_cs

In [13]:
print(len(general_datasets[0]),
      len(general_datasets[1]),
      len(reason_datasets[0]),
      len(reason_datasets[1]),
      len(math_datasets[0]), 
      len(math_datasets[1]),
      len(math_datasets[2]),
      len(humaneval_datasets[0]), 
      len(mbpp_datasets[0]), 
      len(magicoder_datasets[0]))

14042 207865 448 1165 7473 262039 5000 164 378 111183


In [14]:
mix_domains = []

samples = 1000

repeat = 1 if len(general_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(general_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'mmlu',
            'task_label': 0,
            'label': 1,
            'outputs': item['input_correct_responses'][0]
        })

repeat = 1 if len(general_datasets[1]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(general_datasets[1]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'ultrachat',
            'task_label': 1,
            'label': 0,
            'outputs': item['inst'],
        })
# for i, item in enumerate(general_datasets[1]):
#     if i == samples:
#         break
#     mix_domains.append({
#         'inputs': item['input_final_prompts'][0],
#         'task': 'gpqa',
#         'task_label': 1,
#         'label': 0,
#         'outputs': item['input_correct_responses'][0]
#     })
# for i, item in enumerate(general_datasets[2]):
#     if i == samples:
#         break
#     mix_domains.append({
#         'inputs': item['input_final_prompts'][0],
#         'task': 'arc_c',
#         'task_label': 2,
#         'label': 0,
#         'outputs': item['input_correct_responses'][0]
#     })

In [15]:
repeat = 1 if len(reason_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[0]):
        if i == 400:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'gpqa',
            'task_label': 2,
            'label': 1,
            'outputs': item['input_correct_responses'][0]
        })

repeat = 1 if len(reason_datasets[1]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[1]):
        if i == 1000:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'arc_c',
            'task_label': 3,
            'label': 1,
            'outputs': item['input_correct_responses'][0]
        })

In [16]:
repeat = 1 if len(math_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[0]):
        if i == samples:
            break
        # mix_domains.append({
        #     'inputs': item['input_final_prompts'][0],
        #     'task': 'gsm8k',
        #     'task_label': 4,
        #     'label': 2,
        #     'outputs': item['input_correct_responses'][0]
        # })
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'gsm8k',
            'task_label': 4,
            'label': 2,
            'outputs': item['answer']
        })

# repeat = 1 if len(math_datasets[1]) >= samples else 2
# for _ in range(repeat):
#     for i, item in enumerate(math_datasets[1]):
#         if i == 1000:
#             break
#         mix_domains.append({
#             'inputs': item['inst'],
#             'task': 'mathinstruct',
#             'task_label': 5,
#             'label': 2,
#             'outputs': item['output']
#         })

repeat = 1 if len(math_datasets[2]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[2]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['input_final_prompts'][0],
            'task': 'math',
            'task_label': 5,
            'label': 2,
            'outputs': item['input_correct_responses'][0]
        })

In [17]:
repeat = 1 if len(humaneval_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(humaneval_datasets[0]):
        if i == 150:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'humaneval',
            'task_label': 6,
            'label': 3,
            'outputs': item['canonical_solution']
        })

repeat = 1 if len(mbpp_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(mbpp_datasets[0]):
        if i == 300:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'mbpp',
            'task_label': 7,
            'label': 3,
            'outputs': item['code']
        })

repeat = 1 if len(magicoder_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(magicoder_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'magicoder',
            'task_label': 8,
            'label': 3,
            'outputs': item['response']
        })

In [18]:
with open("/data/lihz/datasets/mix_domains/mix_domains_49_v2.jsonl", 'w') as f:
    for item in mix_domains:
        f.write(json.dumps(item) + '\n')

In [19]:
with open("/data/lihz/datasets/mix_domains/mix_domains_49_v1.jsonl", 'r') as f:
    for line in f:
        print(json.loads(line).keys())
        print(json.loads(line)['inputs'])
        print(json.loads(line)['task'])
        print(json.loads(line)['task_label'])
        print(json.loads(line)['label'])
        print(json.loads(line)['outputs'])
        break

dict_keys(['inputs', 'task', 'task_label', 'label', 'outputs'])
<|start_header_id|>user<|end_header_id|>

Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Question: Which of the following conditions will ensure that angular momentum is conserved? I. Conservation of linear momentum II. Zero net external force III. Zero net external torque
A. I and II only
B. I and III only
C. II and III only
D. III only
Your response should end with "The best answer is [the_answer_letter]" where the [the_answer_letter] is one of A, B, C or D.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The best answer is D.<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Question: A pipe full of air is closed at one end. A standing wave is produced in the pipe, causing the pipe to sound a note. Which of the following is a correct statement about the wave’s proper